# Simple Pydantic example with BaseModel
# BaseModel provides
 * Validation: rejects invalid input.
 * Type conversion: can convert "25" to 25 when appropriate.
 * Clear errors: explains which field is missing or invalid.
 * Serialization: converts the object to a dictionary or JSON.
 * Schema generation: describes the expected data structure for APIs and LLM structured outputs.

In [14]:
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    # A type hint is a label in Python that tells humans, editors, and libraries what kind of value a variable, 
    # function input, or output is expected to hold.
    id: int                             # Hint
    # We can set default value as well
    name: str = "Sanjay Mantoor"        # Hint
    age: int                            # Hint
    is_active: bool                     # Hint
    # model_config sets configuration rules for the entire Pydantic model.
    # This is useful when one rule should apply consistently to many fields—rather than repeating 
    # Field(max_length=10) each time.
    model_config = ConfigDict(str_max_length=50)


# Create an instance
# Default user name is assigned
print("Default value - name is used")
user1 = User(id=458026,age=49,is_active=True)
print(user1)
print(user1.model_dump())
print("Overwrite the default value - name")
user2 = User(name="Vanita",id=4580,age=45,is_active=True)
print(user2.model_dump())

Default value - name is used
id=458026 name='Sanjay Mantoor' age=49 is_active=True
{'id': 458026, 'name': 'Sanjay Mantoor', 'age': 49, 'is_active': True}
Overwrite the default value - name
{'id': 4580, 'name': 'Vanita', 'age': 45, 'is_active': True}


## Handling auto conversion example
class User has age of type int. Pydantic can handle conversion to respective data type as possible.

In below example age is passed as string, but Pydantic can handle that conversion.

In [15]:
userConversion = User(name="Sanjay",age="49",id=458026,is_active=True)
print(userConversion.model_dump())

{'id': 458026, 'name': 'Sanjay', 'age': 49, 'is_active': True}


## Data validation error case

In below example, pass age with some string which can't be converted.

In [17]:
try:
    userDataError = User(name="Sanjay",age="22Test",id=458026,is_active=True)
except ValueError as ve:
    print(ve)


1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='22Test', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


## Extra data
By default, Pydantic models won’t error when you provide extra data, and these values will simply be ignored:

In [ ]:
userExtraData = User(name="Sanjay",age=22,id=458026,is_active=True,department="IT")
print(userExtraData.model_dump())

# Extra values can be handled like below

class User_Extra(BaseModel):
    id: int                             
    name: str = "Sanjay Mantoor"        
    age: int                            
    is_active: bool
    # Allow extra data
    # ConfigDict can take ignore ( default option),forbid,allow                     
    model_config = ConfigDict(str_max_length=50,extra='allow') 
    
userExtraData = User_Extra(name="Sanjay",age=22,id=458026,is_active=True,department="IT")
print(userExtraData.model_dump())
print(userExtraData.__pydantic_extra__)    


{'id': 458026, 'name': 'Sanjay', 'age': 22, 'is_active': True}
{'id': 458026, 'name': 'Sanjay', 'age': 22, 'is_active': True, 'department': 'IT'}
{'department': 'IT'}


## Usage of Field

Mechanisms to customize Pydantic model fields: default values, JSON Schema metadata, constraints, etc. Especially useful in AI agents to pass metadata to models.

To do so, the Field() function is used a lot, and behaves the same way as the standard library field() function for dataclasses – by assigning to the annotated attribute

A type hint says what kind of value a field should contain. Field() adds extra rules or metadata for that field.


In [ ]:
from pydantic import BaseModel, Field

class FieldUser(BaseModel):
    name: str = Field(description="Enter User full name")
    age: int = Field(gt=18)

fUser1 = FieldUser(name="Sanjay Mantoor",age=35)
print(fUser1)
try:
    fUser2 = FieldUser(name="Raju",age=10)
except ValueError as ve:
    print(ve)

name='Sanjay Mantoor' age=35
1 validation error for FieldUser
age
  Input should be greater than 18 [type=greater_than, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
1 validation error for FieldUser
name
  Field required [type=missing, input_value={'age': 20}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


### Field - Default values and Validation
* Default values for fields can be provided using the normal assignment syntax or by providing a value to the default argument.
* By default, Pydantic will not validate default values. The validate_default field parameter (or the validate_default configuration value) can be used to enable this behavior

In [36]:
from pydantic import BaseModel, Field

class User(BaseModel):
    age: int = Field(default="twenty six")  # Default value can be assigned but as it is it will not be validated

user1 = User()
print(user1)


# To force the validation use, validate_default=True
class ValidUser(BaseModel):
    age: int = Field(default="twenty six", validate_default=True)

try:
    user2 = ValidUser()
except ValueError as ve:
    print(ve)


age='twenty six'
1 validation error for ValidUser
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty six', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


### Field Aliases

An alias is an alternative name for a field, used when serializing (input) and deserializing(output) data.

There are three ways to define an alias:

	* Field(alias='foo')
	* Field(validation_alias='foo')
	* Field(serialization_alias='foo')

In [44]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name:str = Field(alias='username')

user1 = User(username="Steve")
print(user1.model_dump())
print(user1.model_dump(by_alias=True))


{'name': 'Steve'}
{'username': 'Steve'}


### For validation_alias  and serialization_alias 

In [46]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name:str = Field(validation_alias='username')

user1 = User(username="Steve")
print(user1.model_dump())
# Even with by_alias=True , name is used instead of username
print(user1.model_dump(by_alias=True))


{'name': 'Steve'}
{'name': 'Steve'}


In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name:str = Field(serialization_alias='username')

# It uses name="Raj" instead of username="Raj" in previous cases
user1 = User(name="Raj")
print(user1.model_dump())
# Even with by_alias=True , name is used instead of username
print(user1.model_dump(by_alias=True))

{'name': 'Raj'}
{'username': 'Raj'}


## Demonstrating validation_alias and serialization_alias together

In [ ]:
    from pydantic import BaseModel, Field

    class Book(BaseModel):
        title: str = Field(
            validation_alias="book_title",      # Input will use this
            serialization_alias="bookTitle",    # Output will use this
        )
        author: str = Field(
            validation_alias="author_name",     # Input will use this
            serialization_alias="authorName",   # Output will use this
        )

    backend_data={
        "book_title": "Pydantic Guide",  # Input data is used as per validation_alias
        "author_name": "DataCamp",       # Input data is used as per validation_alias
    }

    book = Book(**backend_data)  # Unpacking dictionary using **
    print(book)
    print(book.title)
    print(book.author)
    print(book.model_dump())
    # Output is bookTitle and authorName as per serialization_alias
    print(book.model_dump(by_alias=True)) 

title='Pydantic Guide' author='DataCamp'
Pydantic Guide
DataCamp
{'title': 'Pydantic Guide', 'author': 'DataCamp'}
{'bookTitle': 'Pydantic Guide', 'authorName': 'DataCamp'}


## Book Model Example

* Required title (string 1-100 chars)
* Required author (string)
* Optional isbn with 13 digits
* price (positive float <= 1000)
* in_stock (boolean, default True)


In [64]:
from pydantic import BaseModel, Field

class Book(BaseModel):
    title: str = Field(min_length=1,max_length=100)
    author: str 
    isbn: str = Field(default=None,pattern=r"^\d{13}$")
    price: float = Field(le=1000,gt=0)
    in_stock: bool = Field(default=True)

valid_book={
    "title":"The Python Crash Course",
    "author":"Eric Matthe",
    "price":900.0
}

book = Book(**valid_book)
print(book.model_dump())

valid_book["isbn"]="1234567891011"

book = Book(**valid_book)
print(book.model_dump())

{'title': 'The Python Crash Course', 'author': 'Eric Matthe', 'isbn': None, 'price': 900.0, 'in_stock': True}
{'title': 'The Python Crash Course', 'author': 'Eric Matthe', 'isbn': '1234567891011', 'price': 900.0, 'in_stock': True}


## Field validators

In its simplest form, a field validator is a callable taking the value to be validated as an argument and returning the validated value. The callable can perform checks for specific conditions (see raising validation errors) and make changes to the validated value (coercion or mutation).


In [73]:
from pydantic import BaseModel, Field, field_validator, ValidationError

class Even(BaseModel):
    number: int = Field(ge=2)
    # First give the data to be validated. In this case number data
    # mode=after means Pydantic first performs its normal validation/conversion, then calls this function.
    @field_validator('number', mode='after')
    @classmethod
    # cls ->refers to the model class, value -> refers to the number data value
    def is_even(cls, value: int) -> int:
        if value % 2 != 0:
            raise ValueError(f"{value} is not even number")
        return value

print(Even(number=4))

try:
    Even(number=3)
except ValidationError as ve:
    print(ve)


number=4
1 validation error for Even
number
  Value error, 3 is not even number [type=value_error, input_value=3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
